# 🎓 Geometric Stream Ablation: MediaPipe & Classical Dlib vs OpenFace

### 🔬 Objective
In this notebook, we evaluate two prominent alternatives to **OpenFace** for the Geometric Stream on your `newly_crafted dataset`:

1. **Google MediaPipe Face Mesh & 52 Blendshapes** (3D Head Pose, Eye Gaze, Facial Muscle Actions)
2. **Classical Geometric Baseline (Dlib style)** (Eye Aspect Ratio [EAR], Mouth Aspect Ratio [MAR], Head Pose [Yaw/Pitch/Roll])

### ⚖️ Fair Comparison Setup:
- ❌ **EfficientNet-B0 is REMOVED**
- ❌ **Transformer Encoder is REMOVED**
- ❌ **'partially engaged' subfolder is OMITTED** (Binary: `engaged` = 0, `disengaged` = 1)
- ✅ **Classification Head is identical**: Temporal Mean Pooling $\to$ FC(128) $\to$ ReLU $\to$ Dropout(0.3) $\to$ FC(2 classes)

---

## 📌 Step 1: Install Dependencies & Download MediaPipe Model Asset

In [ ]:
# Works with modern mediapipe (0.10.30+ and 1.0+)
!pip install -q mediapipe opencv-python-headless torch torchvision scikit-learn matplotlib tqdm pandas numpy
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

import torch
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch Running on: {device}")
print(f"✅ MediaPipe Task Asset Downloaded: {os.path.exists('face_landmarker.task')}")

## 📌 Step 2: Mount Google Drive & Locate `newly_crafted dataset`

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Search candidate paths for the newly crafted dataset
candidates = [
    "/content/drive/MyDrive/newly_crafted dataset",
    "/content/drive/MyDrive/Newly_Crafted_Dataset",
    "/content/drive/MyDrive/newly_crafted_dataset",
    "/content/drive/MyDrive/New_Dataset_Crafted",
    "/content/drive/MyDrive/Student_Engagement_DL/newly_crafted dataset",
    "/content/drive/MyDrive/Student_Engagement_DL/New_Dataset_Crafted"
]

DATASET_ROOT = None
for c in candidates:
    if os.path.exists(c):
        DATASET_ROOT = c
        break

if DATASET_ROOT is None:
    # Fallback to manual path if needed
    DATASET_ROOT = "/content/drive/MyDrive/newly_crafted dataset"

print(f"📁 Dataset root set to: {DATASET_ROOT}")
if os.path.exists(DATASET_ROOT):
    print("Found subfolders:", os.listdir(DATASET_ROOT))
else:
    print("⚠️ Please copy-paste the exact path of your newly_crafted dataset folder above.")

## 📌 Step 3: Scan Video Files (Filtering Engaged vs Disengaged, OMITTING Partially Engaged)

In [ ]:
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

engaged_videos = []
disengaged_videos = []

for root, dirs, files in os.walk(DATASET_ROOT):
    folder_name = os.path.basename(root).lower()
    # STRICT RULE: Completely Omit partially engaged
    if "partial" in folder_name:
        continue
    
    for f in files:
        if f.lower().endswith(video_extensions):
            full_p = os.path.join(root, f)
            if "disengage" in folder_name or "disengaged" in f.lower():
                disengaged_videos.append(full_p)
            elif "engage" in folder_name or "engaged" in f.lower():
                engaged_videos.append(full_p)

print(f"✅ Video Scan Complete:")
print(f"   • Engaged Videos (Class 0):    {len(engaged_videos)}")
print(f"   • Disengaged Videos (Class 1): {len(disengaged_videos)}")
print(f"   • Partially Engaged:           COMPLETELY OMITTED (0)")

## 📌 Step 4: Extract MediaPipe (Blendshapes + 3D Pose) & Classical Geometric Features

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm

os.makedirs("/content/features_mediapipe", exist_ok=True)
os.makedirs("/content/features_classical_dlib", exist_ok=True)

# Initialize MediaPipe Vision Task (Modern API)
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1
)
landmarker = vision.FaceLandmarker.create_from_options(options)

def extract_features(video_path, max_frames=80, sample_fps=4):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 30.0
    step = max(1, int(round(fps / sample_fps)))
    
    f_idx = 0
    mp_seq, dlib_seq = [], []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if f_idx % step == 0:
            h, w, _ = frame.shape
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect(mp_image)
            
            if res.face_landmarks and len(res.face_landmarks) > 0:
                pts = np.array([[lm.x * w, lm.y * h] for lm in res.face_landmarks[0]])
                
                # 1. 3D Head Pose from Transformation Matrix
                if res.facial_transformation_matrixes and len(res.facial_transformation_matrixes) > 0:
                    mat = np.array(res.facial_transformation_matrixes[0])
                    sy = np.sqrt(mat[0, 0]**2 + mat[1, 0]**2)
                    pitch = np.arctan2(mat[2, 1], mat[2, 2])
                    yaw = np.arctan2(-mat[2, 0], sy) if sy > 1e-6 else 0.0
                    roll = np.arctan2(mat[1, 0], mat[0, 0]) if sy > 1e-6 else 0.0
                    pose = [pitch, yaw, roll]
                else:
                    pose = [0.0, 0.0, 0.0]
                
                # 2. Classical Eye Aspect Ratio (EAR) & Mouth Aspect Ratio (MAR)
                ear_l = np.linalg.norm(pts[159] - pts[145]) / (np.linalg.norm(pts[33] - pts[133]) + 1e-6)
                ear_r = np.linalg.norm(pts[386] - pts[374]) / (np.linalg.norm(pts[362] - pts[263]) + 1e-6)
                mar = np.linalg.norm(pts[13] - pts[14]) / (np.linalg.norm(pts[78] - pts[308]) + 1e-6)
                
                # Classical Geometric (Dlib-style) vector (6 dims: Pose + EAR_L + EAR_R + MAR)
                dlib_v = np.array(pose + [ear_l, ear_r, mar], dtype=np.float32)
                dlib_seq.append(dlib_v)
                
                # MediaPipe Blendshapes + 3D Pose (55 dims)
                if res.face_blendshapes and len(res.face_blendshapes) > 0:
                    b_scores = [b.score for b in res.face_blendshapes[0]]
                else:
                    b_scores = [0.0] * 52
                mp_v = np.array(pose + b_scores, dtype=np.float32)
                mp_seq.append(mp_v)
                
                if len(mp_seq) >= max_frames:
                    break
        f_idx += 1
    cap.release()
    if not mp_seq:
        return None, None
    return np.array(mp_seq), np.array(dlib_seq)

# Extract and save features for all videos
all_videos = [(v, 0) for v in engaged_videos] + [(v, 1) for v in disengaged_videos]
print(f"Processing {len(all_videos)} total videos...")

for vid_path, label in tqdm(all_videos, desc="Extracting Geometric Features"):
    base = os.path.splitext(os.path.basename(vid_path))[0]
    mp_feat, dlib_feat = extract_features(vid_path)
    if mp_feat is not None and dlib_feat is not None:
        torch.save({"sequence": torch.from_numpy(mp_feat), "label": label}, f"/content/features_mediapipe/{base}.pt")
        torch.save({"sequence": torch.from_numpy(dlib_feat), "label": label}, f"/content/features_classical_dlib/{base}.pt")

print("\n✅ Feature extraction completed!")
print(f"MediaPipe files saved: {len(os.listdir('/content/features_mediapipe'))}")
print(f"Classical Dlib files saved: {len(os.listdir('/content/features_classical_dlib'))}")

## 📌 Step 5: Train & Evaluate Both Alternatives with the Same Head
Applies **Temporal Representation (Mean Pooling)** across the frames, followed by:
`Linear(D -> 128)` $\to$ `ReLU()` $\to$ `Dropout(0.3)` $\to$ `Linear(128 -> 2)`

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

class GeometricDataset(Dataset):
    def __init__(self, file_paths):
        self.samples = []
        for p in file_paths:
            d = torch.load(p)
            # Temporal Representation (Mean Pooling across frames - Step 5 in Architecture)
            seq = d["sequence"].float()
            pooled = seq.mean(dim=0)  # Shape: [feature_dim]
            self.samples.append((pooled, d["label"]))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

class GeometricClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3, num_classes=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)

def train_and_eval_pipeline(folder_path, model_name):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.pt')]
    labels = [torch.load(f)["label"] for f in files]
    
    train_p, temp_p, _, temp_l = train_test_split(files, labels, test_size=0.30, stratify=labels, random_state=42)
    val_p, test_p = train_test_split(temp_p, test_size=0.50, stratify=temp_l, random_state=42)
    
    train_loader = DataLoader(GeometricDataset(train_p), batch_size=16, shuffle=True)
    val_loader = DataLoader(GeometricDataset(val_p), batch_size=16, shuffle=False)
    test_loader = DataLoader(GeometricDataset(test_p), batch_size=16, shuffle=False)
    
    # Get feature dimension of temporally pooled vector
    sample_pooled = torch.load(files[0])["sequence"].mean(dim=0)
    in_dim = sample_pooled.shape[0]
    
    model = GeometricClassifier(input_dim=in_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    best_val_acc = 0.0
    best_weights = None
    
    for epoch in range(1, 35):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        corr, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                preds = model(x).max(1)[1]
                corr += preds.eq(y).sum().item()
                total += y.size(0)
        val_acc = (corr / total) * 100.0
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.cpu() for k, v in model.state_dict().items()}
            
    # Test evaluation
    model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
    model.eval()
    all_preds, all_y = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            preds = model(x).max(1)[1]
            all_preds.extend(preds.cpu().numpy())
            all_y.extend(y.numpy())
            
    test_acc = accuracy_score(all_y, all_preds) * 100.0
    prec, rec, f1, _ = precision_recall_fscore_support(all_y, all_preds, average='weighted', zero_division=0)
    return best_val_acc, test_acc, rec * 100.0, f1 * 100.0, prec * 100.0

print("Training MediaPipe Model...")
mp_val, mp_test, mp_rec, mp_f1, mp_prec = train_and_eval_pipeline("/content/features_mediapipe", "MediaPipe Face Mesh")

print("Training Classical Dlib Model...")
dl_val, dl_test, dl_rec, dl_f1, dl_prec = train_and_eval_pipeline("/content/features_classical_dlib", "Classical Geometric (Dlib)")

# ------------------------------------------------------------------
# Display Combined Ablation Comparison Table
# ------------------------------------------------------------------
print("\n" + "=" * 95)
print("         📊 FINAL GEOMETRIC STREAM ABLATION COMPARISON TABLE (FOR YOUR PAPER) 📊         ")
print("=" * 95)
headers = ["Geometric Feature Extractor", "Best Val Acc", "Testing Acc", "Recall", "F1-Score", "Precision"]
fmt = "{:<35} | {:<12} | {:<12} | {:<10} | {:<10} | {:<10}"
print(fmt.format(*headers))
print("-" * 95)
print(fmt.format("OpenFace Alone (Previous Run)", "75.00%", "75.00%", "75.00%", "74.42%", "77.47%"))
print(fmt.format("Google MediaPipe Face Mesh Alone", f"{mp_val:.2f}%", f"{mp_test:.2f}%", f"{mp_rec:.2f}%", f"{mp_f1:.2f}%", f"{mp_prec:.2f}%"))
print(fmt.format("Classical Geometric (Dlib) Alone", f"{dl_val:.2f}%", f"{dl_test:.2f}%", f"{dl_rec:.2f}%", f"{dl_f1:.2f}%", f"{dl_prec:.2f}%"))
print("=" * 95)